In [13]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from src import synthetic, features as F

In [14]:
FORECAST_HORIZON = 10
NUM_STORES = 4
NUM_DAYS = 365*4

# Generate sales features

In [15]:

stores = synthetic.store_data(n_stores=NUM_STORES)
sales = synthetic.sales_data(n_stores=NUM_STORES, days=NUM_DAYS)
sales_stores = sales.merge(stores, on='Store', how='left').set_index(['Store', 'Date'])

sales_stores.loc[sales_stores['Sales'] == 0, 'Sales'] = np.nan
sales_stores.ffill(inplace=True)  # Fill NaN values with the last valid observation



In [16]:
""" Past features """
sales = synthetic.sales_data(n_stores=NUM_STORES, days=NUM_DAYS)#.sample(frac=1, random_state=42)
stores = synthetic.store_data(n_stores=NUM_STORES)
sales_stores = sales.merge(stores, on='Store', how='left').set_index(['Store', 'Date'])


lags = [pd.DateOffset(days=i) for i in range(1, 3)]
diffs = [pd.DateOffset(days=i) for i in range(1, 3)]
windows = ['7D', '14D', '30D']


one_day_offset = pd.DateOffset(days=1)
forecast_offset = pd.DateOffset(days=FORECAST_HORIZON)

grouped = sales_stores.reset_index('Store').groupby('Store')['Sales']
x_lag = grouped.apply(lambda x: F.lags(x, lags), include_groups=False)
x_dif = grouped.apply(lambda x: F.diffs(x, diffs, lag=one_day_offset), include_groups=False)
x_win = grouped.apply(lambda x: F.rolling(x, windows, func='mean', lag=one_day_offset), include_groups=False)
x_cal = grouped.apply(lambda x: F.calendar(dates=x.index.to_series(), forecast_offset=forecast_offset), include_groups=False)

x = pd.concat([x_lag, x_dif, x_win, x_cal], axis=1)

In [17]:
pd.concat([sales_stores['Sales'], x], axis=1).sort_index().head(10)

Sales  lag_days_1  lag_days_2  diff_days_1  \
Store Date                                                          
1     2013-01-01  467.320508         NaN         NaN          NaN   
      2013-01-02  467.820508  467.320508         NaN          NaN   
      2013-01-03  451.000000  467.820508  467.320508     0.500000   
      2013-01-04  434.179492  451.000000  467.820508   -16.820508   
      2013-01-05  534.679492  434.179492  451.000000   -16.820508   
      2013-01-06    0.000000  534.679492  434.179492   100.500000   
      2013-01-07  453.000000    0.000000  534.679492  -534.679492   
      2013-01-08  470.820508  453.000000    0.000000   453.000000   
      2013-01-09  571.320508  470.820508  453.000000    17.820508   
      2013-01-10  454.500000  571.320508  470.820508   100.500000   

                  diff_days_2  rolling_mean_7D  rolling_mean_14D  \
Store Date                                                         
1     2013-01-01          NaN              NaN               NaN   
      2013-01-02          NaN       467.320508        467.320508   
      2013-01-03          NaN       467.570508        467.570508   
      2013-01-04   -16.320508       462.047005        462.047005   
      2013-01-05   -33.641016       455.080127        455.080127   
      2013-01-06    83.679492       471.000000        471.000000   
      2013-01-07  -434.179492       392.500000        392.500000   
      2013-01-08   -81.679492       401.142857        401.142857   
      2013-01-09   470.820508       401.642857        409.852564   
      2013-01-10   118.320508       416.428571        427.793446   

                  rolling_mean_30D  year  quarter  month  week_of_month  \
Store Date                                                                
1     2013-01-01               NaN  2013        1      1              2   
      2013-01-02        467.320508  2013        1      1              2   
      2013-01-03        467.570508  2013        1      1              2   
      2013-01-04        462.047005  2013        1      1              3   
      2013-01-05        455.080127  2013        1      1              3   
      2013-01-06        471.000000  2013        1      1              3   
      2013-01-07        392.500000  2013        1      1              3   
      2013-01-08        401.142857  2013        1      1              3   
      2013-01-09        409.852564  2013        1      1              3   
      2013-01-10        427.793446  2013        1      1              3   

                  day_of_week  is_month_start  is_weekend  is_weekday  \
Store Date                                                              
1     2013-01-01            4           False       False        True   
      2013-01-02            5           False        True       False   
      2013-01-03            6           False        True       False   
      2013-01-04            0           False       False        True   
      2013-01-05            1           False       False        True   
      2013-01-06            2           False       False        True   
      2013-01-07            3           False       False        True   
      2013-01-08            4           False       False        True   
      2013-01-09            5           False        True       False   
      2013-01-10            6           False        True       False   

                  is_month_end  
Store Date                      
1     2013-01-01         False  
      2013-01-02         False  
      2013-01-03         False  
      2013-01-04         False  
      2013-01-05         False  
      2013-01-06         False  
      2013-01-07         False  
      2013-01-08         False  
      2013-01-09         False  
      2013-01-10         False